# Phân tích và trực quan hóa dữ liệu Logistics xanh

## Mục tiêu
- Phân tích xu hướng phát thải CO2 theo ngày, tuần và tháng.
- Phân tích số lượng chuyến và lượng phát thải theo trạm xuất phát (Origin Facility).
- So sánh sự phân bố các loại phương tiện vận chuyển.
- Đánh giá sơ bộ lượng phát thải trung bình giữa phương tiện Eco Friendly và Non-Eco Friendly.
- Tạo các biểu đồ tương tác bằng Plotly làm cơ sở cho dashboard của ứng dụng.

## Dữ liệu sử dụng
`cleaned_data.csv` được tạo từ bước tiền xử lý dữ liệu.

In [ ]:
import pandas as pd
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Đọc dữ liệu đã được làm sạch
data = pd.read_csv("cleaned_data.csv")
data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

print("Số dòng:", len(data))
print("Số cột:", len(data.columns))
print("Khoảng thời gian:", data["Date"].min().date(), "đến", data["Date"].max().date())

data.head()

### Nhận xét
Dữ liệu sau khi tiền xử lý được sử dụng làm đầu vào cho module trực quan hóa. Cột `Date` được chuyển sang dạng `datetime` để có thể gom nhóm dữ liệu theo ngày, tuần và tháng.

## 1. KPI tổng quan

In [ ]:
total_co2 = data["Carbon_Emission_kgCO2e"].sum()
avg_co2 = data["Carbon_Emission_kgCO2e"].mean()
total_distance = data["Distance_KM"].sum()
total_package = data["Package_Weight_KG"].sum()
total_trips = len(data)

summary = pd.DataFrame({
    "Metric": [
        "Total CO2 (kgCO2e)",
        "Average CO2/trip (kgCO2e)",
        "Total Distance (km)",
        "Total Package Weight (kg)",
        "Total Trips"
    ],
    "Value": [
        round(total_co2, 2), round(avg_co2, 2),
        round(total_distance, 2), round(total_package, 2), total_trips
    ]
})
summary

### Phân tích
- **Total CO2** phản ánh tổng lượng phát thải của toàn bộ các chuyến vận chuyển.
- **Average CO2/trip** là chỉ số cơ sở để so sánh mức phát thải giữa các nhóm dữ liệu.
- **Total Distance** và **Total Package Weight** mô tả quy mô hoạt động vận chuyển.
- **Total Trips** cho biết tổng số chuyến được ghi nhận.

## 2. Xu hướng phát thải CO2 theo ngày

In [ ]:
daily_emission = (
    data.groupby("Date")["Carbon_Emission_kgCO2e"]
    .sum().reset_index().sort_values("Date")
)
daily_emission.head()

In [ ]:
fig_daily = px.line(
    daily_emission, x="Date", y="Carbon_Emission_kgCO2e",
    title="Daily Carbon Emission Trend", markers=True
)
fig_daily.update_layout(
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="CO2 Emission (kgCO2e)"
)
fig_daily.show()

### Nhận xét biểu đồ 1
Biểu đồ thể hiện sự thay đổi của tổng lượng phát thải CO2 theo từng ngày. Những ngày có mức phát thải tăng cao có thể được xem xét thêm dựa trên số lượng chuyến, điều kiện giao thông và loại phương tiện.

## 3. Xu hướng phát thải CO2 theo tuần

In [ ]:
weekly_emission = (
    data.set_index("Date")["Carbon_Emission_kgCO2e"]
    .resample("W").sum().reset_index()
)
weekly_emission.head()

In [ ]:
fig_weekly = px.bar(
    weekly_emission, x="Date", y="Carbon_Emission_kgCO2e",
    title="Weekly Carbon Emission"
)
fig_weekly.update_layout(
    template="plotly_white",
    xaxis_title="Week",
    yaxis_title="CO2 Emission (kgCO2e)"
)
fig_weekly.show()

### Nhận xét biểu đồ 2
Gom nhóm theo tuần giúp giảm bớt biến động theo ngày và dễ quan sát xu hướng dài hạn hơn. Các tuần có mức phát thải cao có thể tiếp tục được phân tích theo điều kiện giao thông và loại phương tiện.

## 4. Xu hướng phát thải CO2 theo tháng

In [ ]:
# Nếu pandas cũ không hỗ trợ ME, thay "ME" bằng "M".
monthly_emission = (
    data.set_index("Date")["Carbon_Emission_kgCO2e"]
    .resample("ME").sum().reset_index()
)
monthly_emission.head()

In [ ]:
fig_monthly = px.line(
    monthly_emission, x="Date", y="Carbon_Emission_kgCO2e",
    title="Monthly Carbon Emission Trend", markers=True
)
fig_monthly.update_layout(
    template="plotly_white",
    xaxis_title="Month",
    yaxis_title="CO2 Emission (kgCO2e)"
)
fig_monthly.show()

### Nhận xét biểu đồ 3
Biểu đồ tháng cho cái nhìn tổng quát hơn về mức phát thải trong năm và giúp xác định những giai đoạn có tổng phát thải cao để tiếp tục phân tích.

## 5. Phân bổ số lượng chuyến theo trạm xuất phát

In [ ]:
origin_volume = data["Origin_Facility"].value_counts().reset_index()
origin_volume.columns = ["Origin_Facility", "Trip_Count"]
origin_volume = origin_volume.sort_values("Trip_Count", ascending=False)
origin_volume

In [ ]:
fig_origin_volume = px.bar(
    origin_volume, x="Origin_Facility", y="Trip_Count",
    title="Number of Trips by Origin Facility", text_auto=True
)
fig_origin_volume.update_layout(
    template="plotly_white",
    xaxis_title="Origin Facility",
    yaxis_title="Number of Trips"
)
fig_origin_volume.show()

### Nhận xét biểu đồ 4
Biểu đồ cho biết mức độ hoạt động vận chuyển tại từng trạm xuất phát dựa trên số lượng chuyến. Cần kết hợp số chuyến với lượng phát thải để đánh giá tác động môi trường.

## 6. Phát thải CO2 theo trạm xuất phát

In [ ]:
origin_co2 = (
    data.groupby("Origin_Facility")["Carbon_Emission_kgCO2e"]
    .sum().reset_index()
    .sort_values("Carbon_Emission_kgCO2e", ascending=False)
)
origin_co2

In [ ]:
fig_origin_co2 = px.bar(
    origin_co2, x="Origin_Facility", y="Carbon_Emission_kgCO2e",
    title="Carbon Emission by Origin Facility", text_auto=True
)
fig_origin_co2.update_layout(
    template="plotly_white",
    xaxis_title="Origin Facility",
    yaxis_title="CO2 Emission (kgCO2e)"
)
fig_origin_co2.show()

### Nhận xét biểu đồ 5
Biểu đồ xác định các trạm xuất phát có tổng lượng phát thải cao. Tổng phát thải còn phụ thuộc vào số lượng chuyến tại mỗi trạm, vì vậy cần kết hợp với biểu đồ số chuyến.

## 7. Phân bố loại phương tiện vận chuyển

In [ ]:
vehicle_count = data["Vehicle_Type"].value_counts().reset_index()
vehicle_count.columns = ["Vehicle_Type", "Count"]
vehicle_count

In [ ]:
fig_vehicle = px.pie(
    vehicle_count, names="Vehicle_Type", values="Count",
    title="Vehicle Distribution"
)
fig_vehicle.update_layout(template="plotly_white")
fig_vehicle.show()

### Nhận xét biểu đồ 6
Biểu đồ thể hiện cơ cấu sử dụng các loại phương tiện trong dữ liệu logistics. Đây là thông tin nền cho các phân tích chuyên sâu về hiệu quả phương tiện.

## 8. So sánh Eco Friendly và Non-Eco Friendly

In [ ]:
eco_compare = (
    data.groupby("Is_Eco_Friendly")["Carbon_Emission_kgCO2e"]
    .agg(["count", "mean"]).reset_index()
)

eco_compare["Is_Eco_Friendly"] = eco_compare["Is_Eco_Friendly"].map({
    0: "Non-Eco Friendly",
    1: "Eco Friendly"
})

eco_compare

In [ ]:
fig_eco = px.bar(
    eco_compare, x="Is_Eco_Friendly", y="mean",
    title="Average CO2: Eco vs Non-Eco Vehicle", text_auto=True
)
fig_eco.update_layout(
    template="plotly_white",
    xaxis_title="Vehicle Category",
    yaxis_title="Average CO2 (kgCO2e)"
)
fig_eco.show()

### Kiểm chứng sơ bộ
So sánh trên cho biết sự khác biệt về **CO2 trung bình mỗi chuyến** giữa hai nhóm Eco Friendly và Non-Eco Friendly. Kết quả chỉ mang tính mô tả ban đầu, chưa thể kết luận quan hệ nhân quả vì lượng phát thải còn chịu ảnh hưởng bởi khoảng cách, khối lượng hàng, loại phương tiện và điều kiện giao thông.

## 9. Tổng hợp nhanh các kết quả chính

In [ ]:
highest_origin = origin_co2.iloc[0]
highest_trip_origin = origin_volume.iloc[0]
eco_result = eco_compare.set_index("Is_Eco_Friendly")["mean"]

print("=== TỔNG HỢP ===")
print(f"Tổng CO2: {total_co2:,.2f} kgCO2e")
print(f"CO2 trung bình/chuyến: {avg_co2:,.2f} kgCO2e")
print(f"Tổng số chuyến: {total_trips:,}")
print(f"Trạm có tổng CO2 cao nhất: {highest_origin['Origin_Facility']}")
print(f"Số CO2 tại trạm này: {highest_origin['Carbon_Emission_kgCO2e']:,.2f} kgCO2e")
print(f"Trạm có nhiều chuyến nhất: {highest_trip_origin['Origin_Facility']}")
print(f"Số chuyến: {int(highest_trip_origin['Trip_Count']):,}")

if "Eco Friendly" in eco_result.index and "Non-Eco Friendly" in eco_result.index:
    print(f"CO2 TB Eco Friendly: {eco_result['Eco Friendly']:,.2f} kgCO2e")
    print(f"CO2 TB Non-Eco Friendly: {eco_result['Non-Eco Friendly']:,.2f} kgCO2e")

# Kết luận

Qua quá trình trực quan hóa dữ liệu:
- Đã xây dựng các KPI cơ bản về CO2, quãng đường, khối lượng hàng và số chuyến.
- Đã phân tích xu hướng phát thải theo **ngày, tuần và tháng**.
- Đã phân tích **số lượng chuyến** và **tổng phát thải CO2** theo `Origin_Facility`.
- Đã quan sát sự phân bố của các loại phương tiện vận chuyển.
- Đã thực hiện so sánh sơ bộ CO2 trung bình giữa Eco Friendly và Non-Eco Friendly.
- Các biểu đồ sử dụng Plotly nên có khả năng tương tác, hỗ trợ quan sát dữ liệu trực quan.

Kết quả của module này là **phân tích mô tả và trực quan hóa ban đầu**. Các module tiếp theo có thể sử dụng kết quả này để thực hiện kiểm định thống kê, phân cụm phương tiện và phát hiện bất thường.